# PI-CAI nnUNet v2 Professional Pipeline

**6-Channel Input**: T2W + ADC + HBV + 3 One-Hot Zonal Channels

This notebook runs the complete nnUNet v2 pipeline for csPCa segmentation:
1. Mount Google Drive & clone repository
2. Install dependencies (nnUNet v2)
3. Convert PI-CAI data → nnUNet v2 format
4. Plan & preprocess
5. Generate patient-level splits (no data leakage)
6. Train 3D Full-Resolution model

**Auto-Resume**: If Colab disconnects, just re-run all cells. Training resumes from last checkpoint.

---

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset exists
import os
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'
assert os.path.exists(SOURCE_DATA), f'Dataset not found at {SOURCE_DATA}'
assert os.path.exists(os.path.join(SOURCE_DATA, 't2')), 'Missing t2/ directory'
print(f'✅ Dataset found at {SOURCE_DATA}')
print(f'   T2 scans: {len(os.listdir(os.path.join(SOURCE_DATA, "t2")))}')
print(f'   ADC scans: {len(os.listdir(os.path.join(SOURCE_DATA, "adc_reg")))}')
print(f'   HBV scans: {len(os.listdir(os.path.join(SOURCE_DATA, "hbv_reg")))}')
print(f'   Zonal masks: {len(os.listdir(os.path.join(SOURCE_DATA, "zonal_masks")))}')
print(f'   Lesion masks: {len(os.listdir(os.path.join(SOURCE_DATA, "lesion_masks")))}')

## Cell 2: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository (or pull latest if already cloned)
REPO_DIR = '/content/nnu-Net-Baseline'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/HemishJain09/nnu-Net-Baseline.git {REPO_DIR}

print(f'\n✅ Repository ready at {REPO_DIR}')
!ls {REPO_DIR}/*.py

In [ ]:
# Install nnUNet v2 and dependencies
!pip install -q nnunetv2 nibabel scikit-learn tqdm
print('\n✅ All dependencies installed')

## Cell 3: Configure Environment Variables

**Split Storage Architecture:**
- `nnUNet_raw` + `nnUNet_preprocessed` → Local SSD (fast I/O)
- `nnUNet_results` → Google Drive (persistent checkpoints)

In [ ]:
import os

# Local SSD (fast, ephemeral) — for data & preprocessing
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'

# Google Drive (slow, persistent) — for checkpoints & results
RESULTS_DIR = '/content/drive/MyDrive/PI-CAI_nnUNet_Results'
os.environ['nnUNet_results'] = RESULTS_DIR

# Create directories
os.makedirs(os.environ['nnUNet_raw'], exist_ok=True)
os.makedirs(os.environ['nnUNet_preprocessed'], exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Environment Variables Set:')
print(f'  nnUNet_raw:          {os.environ["nnUNet_raw"]}')
print(f'  nnUNet_preprocessed: {os.environ["nnUNet_preprocessed"]}')
print(f'  nnUNet_results:      {os.environ["nnUNet_results"]}')
print('\n✅ Ready')

## Cell 4: Convert PI-CAI Data → nnUNet v2 Format

This copies ~26GB from Google Drive to local SSD and reformats it.

**Handles:**
- 3 imaging modalities → channels 0-2
- Zonal masks → one-hot encoded into channels 3-5
- 207 missing lesion masks → auto-generated blank masks

In [ ]:
!python {REPO_DIR}/convert_picai_to_nnunet.py \
    --source_dir {SOURCE_DATA} \
    --nnunet_raw {os.environ['nnUNet_raw']}

In [ ]:
# Quick verification
import json
dataset_json_path = os.path.join(os.environ['nnUNet_raw'], 'Dataset500_PICAI', 'dataset.json')
with open(dataset_json_path) as f:
    dj = json.load(f)

n_images = len(os.listdir(os.path.join(os.environ['nnUNet_raw'], 'Dataset500_PICAI', 'imagesTr')))
n_labels = len(os.listdir(os.path.join(os.environ['nnUNet_raw'], 'Dataset500_PICAI', 'labelsTr')))

print(f'dataset.json: {json.dumps(dj, indent=2)}')
print(f'\nImages in imagesTr: {n_images} (expected: {dj["numTraining"] * 6})')
print(f'Labels in labelsTr: {n_labels} (expected: {dj["numTraining"]})')

assert n_images == dj['numTraining'] * 6, 'Image count mismatch!'
assert n_labels == dj['numTraining'], 'Label count mismatch!'
print('\n✅ Data conversion verified!')

## Cell 5: nnUNet v2 Plan & Preprocess

This runs nnUNet's automated fingerprint extraction, planning, and preprocessing.

**What it does internally:**
- Extracts dataset fingerprint (spacing, intensity distributions)
- Plans network architecture and patch size
- Resamples all data to target spacing (handles interpolation correctly)
- Saves preprocessed data as `.npz` files

In [ ]:
!nnUNetv2_plan_and_preprocess -d 500 --verify_dataset_integrity -c 3d_fullres

## Cell 6: Generate Patient-Level Cross-Validation Splits

**Critical**: Must run AFTER preprocessing but BEFORE training.

This overwrites nnUNet's random splits with our patient-level GroupKFold splits
to prevent data leakage from the 24 patients with longitudinal scans.

In [ ]:
!python {REPO_DIR}/generate_splits.py \
    --nnunet_raw {os.environ['nnUNet_raw']} \
    --nnunet_preprocessed {os.environ['nnUNet_preprocessed']}

## Cell 7: Train — 3D Full-Resolution, Fold 0

This starts the full nnUNet v2 training.

**Auto-Resume**: If Colab disconnects, re-run all cells.
- Cells 1-4: Will skip conversion (files already exist on SSD check)
- Cell 5: May need to re-run if preprocessed data was lost
- Cell 7: Detects checkpoint on Drive and resumes automatically

**Expected Duration**: ~24-48 hours for full 1000-epoch training on Colab Pro+ (T4/A100)

In [ ]:
import os
import subprocess

FOLD = 0
DATASET_ID = 500
TRAINER = 'nnUNetTrainer'  # Default nnUNet v2 trainer
CONFIG = '3d_fullres'

# Check if a checkpoint exists (for auto-resume)
checkpoint_dir = os.path.join(
    os.environ['nnUNet_results'],
    f'Dataset{DATASET_ID}_PICAI',
    f'{TRAINER}__{CONFIG}__nnUNetPlans',
    f'fold_{FOLD}'
)

checkpoint_exists = os.path.exists(os.path.join(checkpoint_dir, 'checkpoint_final.pth')) or \
                    os.path.exists(os.path.join(checkpoint_dir, 'checkpoint_latest.pth'))

if checkpoint_exists:
    print(f'🔄 Checkpoint found at {checkpoint_dir}')
    print(f'   Resuming training with --c flag...')
    !nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} --c
else:
    print(f'🆕 No checkpoint found. Starting fresh training...')
    print(f'   Fold: {FOLD}')
    print(f'   Config: {CONFIG}')
    print(f'   Results will be saved to: {checkpoint_dir}')
    !nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD}

---

## (Optional) Train Remaining Folds

After Fold 0 completes, run the remaining folds for full 5-fold cross-validation.

In [ ]:
# Uncomment and run each fold one at a time
# !nnUNetv2_train 500 3d_fullres 1
# !nnUNetv2_train 500 3d_fullres 2
# !nnUNetv2_train 500 3d_fullres 3
# !nnUNetv2_train 500 3d_fullres 4

## (Optional) Run Inference on Validation Set

In [ ]:
# After training completes, predict on validation set
# !nnUNetv2_predict -i INPUT_FOLDER -o OUTPUT_FOLDER -d 500 -c 3d_fullres -f 0

# Or use cross-validation predictions across all folds
# !nnUNetv2_find_best_configuration 500 -c 3d_fullres